# STAT 764 · Meeting 20 — Breaking the tools

**Thursday, November 12**

On Tuesday three tools told us what logistic regression and the forest learned. Each one answers a question
about the **model** correctly. Today each one gets a case where that correct answer misleads you about the
**patients**. So before you believe an explanation, ask three questions:

| the question to ask | Tuesday's tool it protects | how that tool misleads | today |
|---|---|---|---|
| **Is something else carrying the same information?** | permutation importance | two look-alike columns can split the credit, so each looks less important than what they share | Section 1 |
| **Were there patients here?** | partial dependence and ICE | the curve keeps going past where there are patients, and the model is guessing there | Section 2 |
| **Am I describing the model, or claiming a cause?** | SHAP values | a push says what moved the model's score, not what would change the patient | Section 3 |

These are the three warnings Lab 9 asks about. The page `breaking.html` in this folder has the same ideas
with pictures. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:                                        # Section 3 and question 3 need it; everything else runs without it
    import shap
except ImportError:
    shap = None
    print("shap is not installed, so Section 3 and question 3 use backup values saved for stays 710 and 18506.\n"
          "To install it: in your clone run `git pull`, then `uv sync`, and restart the kernel (in Colab: !pip install shap).")

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
# Categories as text, counts as decimals: today's tools change one column's values, and decimals keep that exact.
X_all = enc[NUM + CAT].astype({**{c: str for c in CAT}, **{c: float for c in NUM}})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()

# Meeting 16's sample: a random 15,000 of the training patients, with all their stays.
pool = enc["patient_nbr"].iloc[train].unique()
chosen = np.random.default_rng(16).choice(pool, 15_000, replace=False)
rows = train[np.isin(enc["patient_nbr"].iloc[train], chosen)]
X_fit, y_fit = X_all.iloc[rows], y_all.iloc[rows].to_numpy()


def prep(num=NUM):
    """Meeting 9's preprocessing: fill and scale the counts in `num`, one-hot the categories."""
    return ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), list(num)),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), list(CAT)),
    ])


def make_pipe(model, num=NUM):
    """Preprocessing inside the pipeline, any model at the end. `num` lists the count columns to use."""
    return Pipeline([("prep", prep(num)), ("model", model)])


def new_forest():
    """Meeting 16's forest settings."""
    return RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=5, n_jobs=-1, random_state=0)


# Tuesday's two models, with the settings they earned (Meetings 9 and 16).
logistic = make_pipe(LogisticRegression(max_iter=2000)).fit(X_fit, y_fit)
forest = make_pipe(new_forest()).fit(X_fit, y_fit)
p_logistic = logistic.predict_proba(X_test)[:, 1]
p_forest = forest.predict_proba(X_test)[:, 1]
print(f"test: {len(y_test):,} stays · came back {y_test.mean():.1%}")
print(f"AUC on test: logistic regression {roc_auc_score(y_test, p_logistic):.4f} · forest {roc_auc_score(y_test, p_forest):.4f}")

In [ ]:
def shuffle_drop(model, X, y, column, repeats=5, seed=764):
    """Permutation importance for ONE column: the average fall in test AUC when that column is shuffled."""
    rng = np.random.default_rng(seed)
    full = roc_auc_score(y, model.predict_proba(X)[:, 1])
    drops = []
    for _ in range(repeats):
        shuffled = X.copy()
        shuffled[column] = rng.permutation(shuffled[column].to_numpy())   # the column lands on the wrong patients
        drops.append(full - roc_auc_score(y, model.predict_proba(shuffled)[:, 1]))
    return float(np.mean(drops))


def coef_of(pipe, column):
    """A fitted logistic-regression pipeline's coefficient for one count column (in standard deviations)."""
    names = list(pipe[:-1].get_feature_names_out())
    return float(pipe[-1].coef_[0][names.index(f"num__{column}")])


# Backup, used only if shap will not install: the forest's SHAP values for the two stays this notebook explains,
# computed with shap 0.54.1 on the instructor's computer (same data, seeds and settings), in chances. Forests differ
# slightly between computers (an Apple Silicon Mac grows a slightly different one), so yours may not match exactly.
FOREST_SHAP_BACKUP = {
    "base": 0.116069,
    710: {"A1Cresult": 0.003066, "admission_source_id": -0.00171, "admission_type_id": -0.000595,
          "age": -0.00371, "change": 0.002249, "diabetesMed": 0.006766, "discharge_disposition_id": 0.03011,
          "gender": 0.000759, "insulin": -0.002352, "max_glu_serum": -0.000775, "medical_specialty": 0.006176,
          "num_lab_procedures": 0.002097, "num_medications": 0.003766, "num_procedures": 0.003668,
          "number_diagnoses": 0.003383, "number_emergency": -0.000261, "number_inpatient": 0.043772,
          "number_outpatient": -0.000536, "race": -0.002206, "time_in_hospital": 0.001884},
    18506: {"A1Cresult": 0.001658, "admission_source_id": 0.001164, "admission_type_id": -0.001783,
            "age": -0.006945, "change": -0.000143, "diabetesMed": 0.000151, "discharge_disposition_id": 0.037669,
            "gender": 0.000301, "insulin": -0.008717, "max_glu_serum": -6.2e-05, "medical_specialty": -0.012286,
            "num_lab_procedures": 0.001763, "num_medications": -0.002302, "num_procedures": 0.005637,
            "number_diagnoses": -0.00251, "number_emergency": 0.033554, "number_inpatient": 0.127944,
            "number_outpatient": 0.00025, "race": -0.000383, "time_in_hospital": -0.006065},
}


def explain_forest(pipe, patient):
    """SHAP values of the forest pipeline for one patient (a one-row DataFrame), from shap's TreeSHAP.

    Returns (pushes, base) in CHANCES, not log-odds: a forest averages its trees' chances, so its pushes add up in
    chances. base is the forest's average chance over its training patients; base + pushes.sum() is its chance
    for this patient. Without shap it falls back to FOREST_SHAP_BACKUP, for stays 710 and 18506 only.
    """
    if shap is None:
        stay = X_test.index.get_loc(patient.index[0])            # the stay's position among the test stays
        if stay not in FOREST_SHAP_BACKUP:
            raise RuntimeError(f"shap is not installed, and the backup has only stays 710 and 18506, not {stay}. "
                               "In your clone run `git pull`, then `uv sync`, and restart the kernel.")
        print(f"(shap is not installed: stay {stay}'s values are saved from the instructor's computer. Forests differ"
              " slightly between computers, so they may not add up exactly to your forest's chance.)")
        pushes = pd.Series(FOREST_SHAP_BACKUP[stay])
        return pushes.sort_values(key=abs, ascending=False), FOREST_SHAP_BACKUP["base"]
    pre, model = pipe[:-1], pipe[-1]
    z = pre.transform(patient)
    z = z.toarray() if hasattr(z, "toarray") else np.asarray(z)
    explainer = shap.TreeExplainer(model)                    # TreeSHAP: exact SHAP values from the trees themselves
    values = explainer.shap_values(z)
    # A classifier gets values for both classes (did not come back, came back); keep "came back".
    values = values[1] if isinstance(values, list) else (values[..., 1] if np.ndim(values) == 3 else values)
    # The model sees one-hot pieces (age_[80-90), age_[70-80), ...); add each original column's pieces back together.
    names = pre.get_feature_names_out()                      # e.g. num__number_inpatient, cat__age_[80-90)
    column = [n.split("__", 1)[1] if n.startswith("num__") else
              max((c for c in CAT if n.split("__", 1)[1].startswith(c + "_")), key=len) for n in names]
    pushes = pd.Series(np.ravel(values), index=column).groupby(level=0).sum()
    expected = np.ravel(explainer.expected_value)
    base = float(expected[1] if len(expected) > 1 else expected[0])
    return pushes.sort_values(key=abs, ascending=False), base


print("helpers ready: shuffle_drop, coef_of, explain_forest")

## 1. Look-alike columns: importance splits the credit

Meeting 16's studio added a column that almost copies prior stays: `inpatient_stays_last_year`, prior stays plus
a little noise, as a second records system might report them. The two agree to a correlation of 0.97. Below,
the forest is refitted with the copy added, and we ask Tuesday's question again: how much does each column
matter?

Two measures. **Shuffling** each column (permutation importance, Tuesday's tool) gives the usual importance table:
each column's fall in test AUC when it is shuffled. It measures what this fitted model relies on. **Dropping** a
column refits the model without it, so the model can lean on whatever is left. It measures what a column adds that the
others cannot replace. About a minute and a half: two importance tables and three extra forests.

In [ ]:
COPY = "inpatient_stays_last_year"
noise = np.random.default_rng(3)                                  # Meeting 16's seed and order
X_fit2, X_test2 = X_fit.copy(), X_test.copy()
X_fit2[COPY] = X_fit2["number_inpatient"] + noise.normal(0, 0.3, len(X_fit2))
X_test2[COPY] = X_test2["number_inpatient"] + noise.normal(0, 0.3, len(X_test2))
NUM2 = NUM + [COPY]                                               # the counts, plus the look-alike
print(f"correlation of the two columns: {np.corrcoef(X_fit2['number_inpatient'], X_fit2[COPY])[0, 1]:.2f}")

forest2 = make_pipe(new_forest(), NUM2).fit(X_fit2, y_fit)
auc2 = roc_auc_score(y_test, forest2.predict_proba(X_test2)[:, 1])

# Shuffling: the importance table, Tuesday's forest beside the forest with the copy.
imp_alone = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=764)
imp_copy = permutation_importance(forest2, X_test2, y_test, scoring="roc_auc", n_repeats=5, random_state=764)
table = pd.DataFrame({"no copy (Tuesday)": pd.Series(imp_alone.importances_mean, index=X_test.columns),
                      "with the copy": pd.Series(imp_copy.importances_mean, index=X_test2.columns)})
print("\nthe forest's importance table (AUC drop when each column is shuffled):")
print(table.sort_values("with the copy", ascending=False).head(6).round(4).to_string())

# Dropping: refit without the column(s), and compare test AUC with the forest that has both.
print("\ndropping and refitting (AUC drop from the forest with both):")
for label, gone in (("without the copy", [COPY]), ("without prior stays", ["number_inpatient"]),
                    ("without both", [COPY, "number_inpatient"])):
    kept = [c for c in NUM2 if c not in gone]
    refit = make_pipe(new_forest(), kept).fit(X_fit2.drop(columns=gone), y_fit)
    print(f"  {label:20s} {auc2 - roc_auc_score(y_test, refit.predict_proba(X_test2.drop(columns=gone))[:, 1]):+.4f}")

Alone, prior stays tops the forest's importance table at 0.057. Add the look-alike and it falls to 0.018, with the copy
at 0.017, because shuffling one leaves the other in place. Where the patient was discharged to barely moves. Drop either
and the model loses nothing (the drops round to 0.00);
drop **both** and it loses 0.036. Neither tool is wrong: each answers "what does the model lose if this one
column breaks?" But a table that says "prior stays: 0.02" invites the wrong reading, that prior stays barely
matters. With look-alikes, judge them **together**.

## 2. Thin data: the curve keeps going after the patients run out

Tuesday's partial dependence stopped at 7 prior stays. Here it runs to 15, beside how many **training** stays
sit at each value. A model can only learn from patients it has seen.

In [ ]:
def curves(model, patients, column, values):
    """Each patient's predicted chance as `column` is set to each of `values`, everything else unchanged.

    One row per patient (an ICE curve), one column per value; the column means are the partial dependence.
    """
    out = []
    for v in values:
        changed = patients.copy()
        changed[column] = v                              # everyone gets the same value of this one column
        out.append(model.predict_proba(changed)[:, 1])
    return np.array(out).T


ice_patients = X_test.sample(200, random_state=19)       # Tuesday's 200 test stays
wide = np.arange(0, 16, dtype=float)                     # prior stays 0, 1, ..., 15
pd_logistic = curves(logistic, ice_patients, "number_inpatient", wide).mean(axis=0)
pd_forest = curves(forest, ice_patients, "number_inpatient", wide).mean(axis=0)
counts = X_fit["number_inpatient"].value_counts().reindex(wide, fill_value=0)

print(pd.DataFrame({"training stays": counts.to_numpy(), "logistic (partial dependence)": pd_logistic.round(3),
                    "forest (partial dependence)": pd_forest.round(3)}, index=wide.astype(int)).to_string())
# How many training stays sit out where the curves keep going (a few have more than 15):
print(f"\ntraining stays with 8 or more prior stays: {(X_fit['number_inpatient'] >= 8).sum()}"
      f" · 10 or more: {(X_fit['number_inpatient'] >= 10).sum()} · 15 or more: {(X_fit['number_inpatient'] >= 15).sum()}")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(wide, counts / counts.max() * 0.9, color="0.85", label="training stays at this value (scaled)")
ax.plot(wide, pd_logistic, color="black", lw=2.5, label="logistic regression")
ax.plot(wide, pd_forest, color="black", lw=2.5, ls="--", label="forest")
ax.set(xlabel="prior inpatient stays", ylabel="predicted chance (partial dependence)", ylim=(0, 0.95))
ax.legend(loc="upper left")
plt.show()

Past about 7 prior stays the bars vanish: 96 training stays had 8 or more, 40 had 10 or more, 8 had 15 or more.
Out there logistic regression keeps its straight line in log-odds and climbs to 83%; the forest can only average
the few patients it saw and levels off near 31%. **Both curves are the model talking, not the data.** The fix is
to draw them only where there are patients, and to say so when one of yours lands out there.

## 3. Not a cause: a push is about the model

The `shap` package computes SHAP values for the forest with its **TreeSHAP** algorithm. They mean what
Tuesday's did, a push measured from a typical patient, with two differences: a forest averages chances, so its
pushes add up in **chances**, not log-odds; and it starts from its average chance over the training patients
(11.6%), not Tuesday's patient with every column at its average (10.4%).

Take Tuesday's biggest disagreement, stay 18506: 12 prior stays, logistic regression 91.8%, the forest 28.5%.

In [ ]:
# Double brackets keep a one-row DataFrame, which is what the pipeline expects.
contributions_18506, base_forest = explain_forest(forest, X_test.iloc[[18506]])
print(f"stay 18506: {X_test['number_inpatient'].iloc[18506]:.0f} prior stays · forest {p_forest[18506]:.1%}"
      f" · logistic regression {p_logistic[18506]:.1%} · came back: {bool(y_test[18506])}")
print(f"the forest's typical patient: {base_forest:.1%}")
print("\nthe forest's largest pushes, in CHANCE (+0.10 is ten points):")
print(contributions_18506.head(6).round(3).to_string())
# The check SHAP always passes: the starting chance plus every push is the model's own chance.
print(f"\n{base_forest:.3f} + pushes {contributions_18506.sum():+.3f} = {base_forest + contributions_18506.sum():.3f}"
      f" (the forest says {p_forest[18506]:.3f})")

# Logistic regression's push for the same 12 prior stays, as Tuesday's explain_linear computed it: the coefficient
# times how far this patient is from the average, in standard deviations, on the log-odds scale.
z = (X_test["number_inpatient"].iloc[18506] - X_fit["number_inpatient"].mean()) / X_fit["number_inpatient"].std(ddof=0)
print(f"logistic regression's push for prior stays: {coef_of(logistic, 'number_inpatient') * z:+.2f} log-odds")

Twelve prior stays push the forest up about 13 points. They push logistic regression up +3.03 in log-odds, which on
its own would take Tuesday's 10.4% typical patient to about 71%. Both explanations are exact; they explain
**different models**.

What neither one says is what would happen to the **patient**. "Prior stays push the score up" is a fact about
the model. "Preventing earlier stays would lower this patient's risk" is a claim about the hospital, and nothing
in the data tested it. Tuesday's exit ticket asked whether preventing earlier stays would lower a patient's chance:
eight of nine of you said no, because the model using prior stays is not prior stays causing readmission. The studio
shows why that matters.

## Studio

**Everything you need is in this notebook.** Sections 1–3 break each of Tuesday's tools once. The studio breaks
them again, on the other model or against what actually happened.

| | what you practice | about |
|---|---|---|
| **Question 1** | reading coefficients when two columns look alike | 15 minutes |
| **Question 2** | checking a model's chances where the data runs thin | 15 minutes |
| **Question 3** | telling a push in the model from a cause in the hospital | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24 hours after
class.

**How to work.**

1. At the start of class, **Run All**: Section 1 takes about a minute and a half. The questions use `logistic`,
   `forest`, `X_fit`, `y_fit`, `X_test`, `y_test`, `p_logistic`, `p_forest`, `X_fit2`, `X_test2`, `NUM2`, `COPY`,
   `make_pipe`, `shuffle_drop`, `coef_of` and `explain_forest`.
2. Before each question, **write your guess** as a comment in its cell, and keep it when the answer disagrees.
   The Compare uses it.
3. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
4. Save and upload to Canvas under "Studio notebook — Day 20" by **Friday, November 13, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `X_fit2`, `NUM2` or `shuffle_drop` | Run the notebook from the top (Run All). |
| `ValueError: ... not in list` from `coef_of` | The column is not in that model: `logistic` has no copy; use the model fitted on `X_fit2`. |
| `ValueError: Some column names are not columns of the dataframe` when refitting | Give `make_pipe` the columns you kept: `make_pipe(model, kept)`, with `kept` the counts left in. |
| `TypeError: '<' not supported between instances of 'int' and 'str'` | Codes are text: set the discharge to `"1"`, in quotes, not `1`. |
| `ValueError: array length 24706 does not match index length 21352` | Question 2 groups the **test** stays: `pd.cut(X_test[...])`, not `X_fit`. |
| The share aged 70+ comes out 0 | Ages are text ranges: `"[70-80)"`, `"[80-90)"`, `"[90-100)"`. |
| `(shap is not installed: stay ...'s values are saved ...)` | Fine: Section 3 and question 3 work from saved values. To install shap: `git pull`, then `uv sync` in your clone; restart the kernel and Run All. |
| Question 3(c) shows 0 stays or `NaN` | Codes are text: compare with `== "1"`, not `== 1`. |
| Question 1(d)'s drops look wrong | Measure them from `logistic2`'s own AUC, `auc_l2`, not Section 1's `auc2` (the forest's). |
| `explain_forest` fails on one stay | Pass a one-row DataFrame, `X_test.iloc[[i]]`, with double brackets. |
| Question 2's groups print as 0, 1, 10+, 2, … | Keep the groups' order: put `group.reset_index(drop=True)` in your table, not `group.to_numpy()`. |

### 1. Look-alike columns, in logistic regression

- **(a)** Fit `logistic2 = make_pipe(LogisticRegression(max_iter=2000), NUM2).fit(X_fit2, y_fit)`: Tuesday's
  logistic regression with the look-alike added.
- **(b)** Print four numbers with `coef_of`: prior stays in `logistic` (no copy); prior stays and `COPY` in
  `logistic2`; and the sum of those two. Each coefficient is the change in log-odds for one standard deviation more
  of that count, holding the other columns fixed.
- **(c)** `shuffle_drop` on `logistic2` with `X_test2` and `y_test`, for prior stays and for `COPY`.
- **(d)** First `auc_l2 = roc_auc_score(y_test, logistic2.predict_proba(X_test2)[:, 1])` (Section 1's `auc2` is the
  forest's). Then copy Section 1's drop-and-refit loop, with `LogisticRegression(max_iter=2000)` in place of
  `new_forest()` and `auc_l2` in place of `auc2`: without `COPY`, without prior stays, without both.

Guess first: will logistic regression split the credit between the two the way the forest did?

**Answer in three sentences:** what would someone reading `logistic2`'s coefficients one at a time conclude about
stays last year, and what does the sum say? What does (c) say `logistic2` relies on, and (d) what each column adds
that the other cannot replace? Is that the forest's pattern?

### 2. Thin data: check the chances against what happened

Section 2 set every patient to the same count. Here, take the real test stays at each count, and compare each
model's average chance with the share who came back. On Tuesday, for the 592 stays where the models disagreed, 27.9%
came back; the forest said 24.4% and logistic regression 35.8%. Does "the forest is closer" hold everywhere?

- **(a)** Put each test stay in a group by prior stays: 0, 1, 2, 3, 4, 5–6, 7–9 and 10 or more, with
  `pd.cut(X_test["number_inpatient"], [-1, 0, 1, 2, 3, 4, 6, 9, 99], labels=["0", "1", "2", "3", "4", "5-6", "7-9", "10+"])`.
- **(b)** One table, a row per group: the number of test stays, the share who came back, and each model's
  average chance (`p_logistic`, `p_forest`) for those stays.
- **(c)** For every group, a rough 95% range for the share who came back, as two more columns: the share
  ± 2 × √(share × (1 − share) / number of stays).

Guess first: for the patients with 10 or more prior stays, which model's average will be nearer what happened?

**Answer in two or three sentences:** in which groups are both models within about 2 points of what happened?
Past 4 prior stays, where does each model fall outside the range, and in which direction?

### 3. Not a cause: what the hospital should not do with a push

Tuesday's stay 710 was discharged to another inpatient care facility (code `"5"`), and that discharge pushed
logistic regression's score up the most.

- **(a)** `explain_forest(forest, X_test.iloc[[710]])`: the forest's pushes for stay 710 and its typical patient.
  Does the discharge push it up too?
- **(b)** `home = X_test.iloc[[710]].copy()`, then `home["discharge_disposition_id"] = "1"` (home; codes are
  text). Print each model's chance for stay 710 as it is (`X_test.iloc[[710]]`) and as changed (`home`).
- **(c)** Among the **training** stays, compare three discharge codes: `"1"` home, `"5"` another inpatient facility,
  `"3"` a skilled nursing facility. Start from
  `d = X_fit.assign(came_back=y_fit, aged_70=X_fit["age"].isin(["[70-80)", "[80-90)", "[90-100)"]))`, keep the
  rows whose `discharge_disposition_id` is one of the three codes, and group by it. For each code: the number of
  stays, the share who came back, the average `time_in_hospital`, the average `number_diagnoses`, and the share
  aged 70 or older.

Guess first: will the patients sent to facilities look sicker than those sent home?

**Answer in three or four sentences:** (b) says that sending stay 710 home lowers the model's chance. The
hospital asks: should we send fewer patients to other facilities? What does (c) tell you about who gets sent
there, and what would you need before you could answer the hospital?

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Tuesday, November 17), built from the notebooks you upload. So
leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: the four coefficient numbers, and the three drops.
3. Question 2: your table, and where each model parts from what happened.
4. Question 3: what you would tell the hospital.

## Exit ticket

> Rudin (2019), this week's reading, argues that when a model you can read is about as accurate, you should use
> it instead of explaining a black box. Our logistic regression tied the forest. In two or three sentences: which
> would you hand the hospital for the \$150 visit? Start from the decision, the first of Tuesday's five steps for
> picking a model: do the patients the two models disagree about most end up with a different decision? Then give
> one warning to go with your choice.

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook — Day 20". Your
two studio points need this answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Rudin (2019)**, "Stop explaining black box machine learning models for high stakes decisions and use
interpretable models instead," *Nature Machine Intelligence* 1:206–215 — this week's reading.

📗 **Molnar, *Interpretable Machine Learning*** (free online) — the chapters on partial dependence and
accumulated local effects (ALE). ALE plots are built to cope with look-alike columns, which partial dependence
handles badly.

📗 **Conditional permutation importance** (Strobl et al., 2008) — shuffle a column only among patients who look
alike on its look-alikes, so the credit is not simply split.

🎓 **Causal inference** — what it takes to answer the hospital's question in question 3: an experiment, or
careful assumptions about why patients were sent where. Hernán and Robins, *Causal Inference: What If* (free
online), is the standard start. It needs a course of its own.